# Generación de Datos V9 con BSG (Beam Search Greedy)

Pipeline idéntico al de V9 victorioso, pero usando **BSG (w=5)** como evaluador de movimientos en lugar de greedy puro.

- **Adapter:** `EnrichedStackMatrix5DAdapter` (mismo que V9)
- **Solver:** BSG con beam width `w=5`
- **Salida:** archivos `.data` con sufijo `_V9_BSG`

BSG explora un beam de candidatos en cada nivel y evalúa cada uno con greedy, encontrando soluciones de menor costo que greedy puro. Esto produce **etiquetas de mayor calidad** para el entrenamiento.

## 1. Setup

In [ ]:
# Solo en Colab: clonar el repositorio
# !git clone https://github.com/felipe-astudillo-s/CPMP-Transformer.git
# %cd CPMP-Transformer

In [ ]:
import sys
import os

# Si el notebook se abre desde notebooks/, sube un nivel a CPMP-Framework/
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print('CWD:', os.getcwd())
print('src:', src_path)

In [ ]:
# Compilar el solver FRG (requiere g++ — en Windows usar WSL o ejecutar en Colab)
# Si el binario 'frg' ya existe y funciona, puedes saltarte esta celda.
!g++ Codigo_C_solver/Greedy.cpp Codigo_C_solver/Layout.cpp Codigo_C_solver/Bsg.cpp Codigo_C_solver/main_cpmp.cpp -o Codigo_C_solver/frg -O3 -std=c++11
!chmod +x Codigo_C_solver/frg
print('Solver listo:', os.path.abspath('Codigo_C_solver/frg'))

In [ ]:
# Opcional: montar Google Drive para guardar los .data generados
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_DEST = '/content/drive/MyDrive/CPMP_Data/V9_BSG'
# os.makedirs(DRIVE_DEST, exist_ok=True)
# print(f'Carpeta Drive lista: {DRIVE_DEST}')

## 2. Generación de instancias y datos con BSG

In [ ]:
from generation.instances import generate_instances
from generation.data import generate_data
from generation.adapters import EnrichedStackMatrix5DAdapter, DefaultMovesAdapter
from settings import DATA_FOLDER

# Mismas configuraciones que V9 victorioso
# Para un baseline rapido, usa solo las 2 primeras (100k muestras, ~4-6h en Colab)
# Para el pipeline completo, descomenta las 4 configuraciones
configuraciones = [
    # (H, S, N, amount)
    (5, 4, 15, 50000),    # E4-15-H5_V9_BSG  — tableros pequeños
    (7, 5, 25, 50000),    # E5-25-H7_V9_BSG  — benchmark intermedio (50k para baseline)
    # (10, 6, 45, 50000), # E6-45-H10_V9_BSG — difíciles (lento con BSG)
    # (6, 7, 30, 50000),  # E7-30-H6_V9_BSG  — anchos
]

BSG_BEAMS = 5   # Beam width para BSG
r_desorden = 50
max_steps = 50
seed = 42

print(f'Iniciando pipeline V9-BSG (beams={BSG_BEAMS})...\n')

for H, S, N, amount in configuraciones:
    filename = f'E{S}-{N}-H{H}_V9_BSG'
    print(f'--- Configuracion: {S} Stacks, {N} Contenedores, Altura {H} ---')

    print(f'Generando {amount} instancias brutas...')
    generate_instances(filename, H, S, N, amount, r_desorden, seed)

    print(f'Resolviendo con BSG (w={BSG_BEAMS}) y empaquetando datos 5D...')
    layout_adapter = EnrichedStackMatrix5DAdapter()
    moves_adapter = DefaultMovesAdapter()

    generate_data(filename, H, max_steps, layout_adapter, moves_adapter, beams=BSG_BEAMS)
    print(f'Finalizado: {DATA_FOLDER / (filename + ".data")}\n')

    # Opcional: copiar a Drive
    # import shutil
    # shutil.copy(str(DATA_FOLDER / f'{filename}.data'), f'{DRIVE_DEST}/{filename}.data')
    # print(f'Subido a Drive: {DRIVE_DEST}/{filename}.data\n')

print('Generacion V9-BSG terminada!')

## 3. Entrenamiento del modelo V9 con datos BSG

In [ ]:
from preprocessing.dataset import load_dataset
from torch.utils.data import ConcatDataset

# Cargar los datasets generados (ajusta los nombres si solo usaste las 2 primeras configs)
datasets = []
data_files = [
    'E4-15-H5_V9_BSG.data',
    'E5-25-H7_V9_BSG.data',
    # 'E6-45-H10_V9_BSG.data',
    # 'E7-30-H6_V9_BSG.data',
]

for fname in data_files:
    datasets.append(load_dataset(fname))

data = ConcatDataset(datasets)
print(f'Dataset combinado: {len(data)} instancias.')

In [ ]:
from models.cpmp_transformer_v9 import CPMPTransformer

# Mismos hiperparámetros que v9_Original
model = CPMPTransformer(
    H=12,
    C_dim=2,
    X_dim=5,
    d_model=64,
    nhead=4,
    num_layers=4,
    ff_dim_multiplier=2,
    dropout=0.3
)

In [ ]:
from training.training import train, Accuracy

epochs = 30
train_size = int(len(data) * 0.8)
test_size = len(data) - train_size
batch_size = 128
learning_rate = 4e-4
weight_decay = 1e-3
patience = 10
metrics = [Accuracy()]
seed = 42

model = train(model, epochs, data, train_size, test_size, batch_size, learning_rate, weight_decay, patience, metrics, seed)

In [ ]:
from training.training import save_model

save_model(model, 'v9_BSG')